# GIAI ĐOẠN 5: KIỂM ĐỊNH GIẢ THUYẾT THỐNG KÊ (H1–H4) & MÔ HÌNH HÓA HỌC MÁY (M0a–M7 + EX-ANTE)
**Đề tài:** Phân tích Biểu hiện Định giá Động (*Dynamic Surge Pricing*) và Dự báo Giá báo API trên Nền tảng Gọi xe Công nghệ (Uber & Lyft Boston, MA)  
**Mã đề tài:** `PRJ-EXT-01` | **Nhóm thực hiện:** Nhóm 02 — Trần Thanh Huy (`2521001060`) & Lê Huy Thắng (`2521001092`)  
**Môi trường thực thi:** Python `3.11.9` | `pandas 3.1.1` | `numpy 2.4.6` | `scipy 1.17.1` | `statsmodels 0.15.0` | `scikit-learn 1.9.1`

---
## PHẦN I: KIỂM ĐỊNH GIẢ THUYẾT THỐNG KÊ CHUYÊN SÂU & ĐỘ LỚN HIỆU ỨNG (H1 -> H4)
- **Nguyên tắc phương pháp luận:** Với cỡ mẫu lớn ($N = 270,898$; `232,861` mẫu surge quan sát thực; `152,043` mẫu khí tượng sạch lỗi cảm biến mưa), mọi kiểm định tham số (*Welch's t-test, One-way ANOVA, Chi-Square, Wald Z/F-test*) đều đi kèm **kiểm định phi tham số đối chứng** (*Mann-Whitney U, Kruskal-Wallis H, G-test*), **độ lớn hiệu ứng**, sai số chuẩn cụm hai cấp và hiệu chỉnh đa kiểm định Holm–Bonferroni & BH-FDR.

In [1]:
import sys
from pathlib import Path

import pandas as pd

BASE_DIR = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(BASE_DIR / "src") not in sys.path:
    sys.path.insert(0, str(BASE_DIR / "src"))

from features import load_master_dataset  # noqa: E402
from stats_tests import test_h1_weather_price  # noqa: E402

# Kiểm chứng trực tiếp H1 từ mã nguồn trên master_rideshare.parquet
df_master_live = load_master_dataset()
h1_live = test_h1_weather_price(df_master_live)[0]
print(
    f"[Live Verify H1a] n_rain={h1_live['n_group1']:,}, "
    f"n_dry={h1_live['n_group2']:,}, "
    f"t={h1_live['parametric_stat']:.4f}, "
    f"p={h1_live['parametric_pvalue']:.4f}, "
    f"d={h1_live['effect_size_value']:+.5f}\n"
)

hyp_df = pd.read_csv(BASE_DIR / "reports/tables/stage5_hypothesis_tests.csv")
cols_show = [
    "hypothesis_id",
    "parametric_test",
    "parametric_stat",
    "parametric_pvalue",
    "effect_size_name",
    "effect_size_value",
    "statistical_decision",
    "practical_significance",
]
print(hyp_df[cols_show].to_string(index=False))

2026-11-28 03:42:25,566 - INFO - Đã nạp master_rideshare.parquet: 270,898 dòng x 32 cột.


[Live Verify H1a] n_rain=24,495, n_dry=127,548, t=-0.1365, p=0.8915, d=-0.00095

                      hypothesis_id               parametric_test  parametric_stat  parametric_pvalue                             effect_size_name  effect_size_value         statistical_decision                                                                                            practical_significance
                      H1a_All_Rides                Welch's t-test          -0.1365           0.891453                                    Cohen's d          -0.000950 Chưa đủ bằng chứng bác bỏ H0                                                                                        Không đáng kể (|d| < 0.01)
                      H1b_Uber_Only                Welch's t-test           0.0360           0.971280                                    Cohen's d           0.000350 Chưa đủ bằng chứng bác bỏ H0                                                                                        Không đáng kể (|d| 

### Biểu giá Kinh tế lượng 12 Phân khúc (Statsmodels OLS với Sai số chuẩn Bền vững HC3) & Điểm cắt Biểu giá
- **Kết quả Kiểm định Wald H4a (`UberX` vs `Lyft`, Đặc tả A trên `X_train` có surge thực):**
  - `UberX`: Giá mở cửa $\hat{\beta}_0 = \$6.47$, đơn giá mỗi dặm $\hat{\beta}_1 = \$1.51$/dặm (`1.5081`, khớp `stage5_ols_coefficients.csv`).
  - `Lyft`: Giá mở cửa $\hat{\beta}_0 = \$5.40$, đơn giá mỗi dặm $\hat{\beta}_1 = \$1.84$/dặm.
  - **Điểm cắt cự ly (*Crossing Distance*):** Đặc tả A $d^* = 3.21$ dặm (CI cụm `[2.96, 3.46]` dặm); Đặc tả B thuần không surge $d^* = 5.17$ dặm. Riêng `UberXL` vs `Lyft XL` cắt tại `3.75` dặm ở Đặc tả A nhưng dịch ra `8.32 > 7.86` dặm ở Đặc tả B (Lyft XL rẻ hơn toàn miền khi không surge).

In [2]:
ols_df = pd.read_csv(BASE_DIR / "reports/tables/stage5_ols_coefficients.csv")
ols_cols = [
    "cab_type",
    "service_tier",
    "n_train",
    "base_fare_b0_usd",
    "base_fare_se_hc3",
    "per_mile_rate_b1_usd",
    "per_mile_se_hc3",
    "r2_within_tier",
]
print(ols_df[ols_cols].to_string(index=False))

cab_type service_tier  n_train  base_fare_b0_usd  base_fare_se_hc3  per_mile_rate_b1_usd  per_mile_se_hc3  r2_within_tier
    Lyft       Shared    11367            3.3439            0.0349                1.2377           0.0175          0.4045
    Lyft         Lyft    11340            5.4039            0.0298                1.8387           0.0145          0.7928
    Lyft      Lyft XL    11228            7.4918            0.0550                3.4114           0.0276          0.8327
    Lyft          Lux    11238            8.5244            0.0623                4.0487           0.0319          0.8515
    Lyft    Lux Black    11386           11.7449            0.0792                4.9637           0.0397          0.8458
    Lyft Lux Black XL    11274           20.7326            0.1177                5.0902           0.0579          0.7230
    Uber     UberPool    17043            6.1400            0.0263                1.1862           0.0127          0.4463
    Uber          WAV   

---
## PHẦN II: TINH CHỈNH SIÊU THAM SỐ, 5-FOLD CROSS-VALIDATION & SO SÁNH 10 CẤU HÌNH MÔ HÌNH + EX-ANTE
### Bước 2.1. Kết quả Tinh chỉnh Siêu tham số (Chỉ trên tập Train `X_train`, `n = 186,288`)
- **M3 (`RidgeCV`):** $\alpha^* = 0.1$.
- **M4 (`LassoCV`):** $\alpha^* = 5.6 \times 10^{-5}$ (`0.000056`) (tối ưu `precompute=True` + `warm_start=True`).
- **M5 (`DecisionTree`):** `{'max_depth': 15, 'min_samples_leaf': 15, 'min_samples_split': 10}`.
- **M6 (`RandomForest`):** `{'max_depth': 20, 'max_features': 0.7, 'min_samples_leaf': 5}`.
- **M7 (`HistGradientBoosting`):** `{'learning_rate': 0.08, 'max_leaf_nodes': 31, 'min_samples_leaf': 20, 'l2_regularization': 5.0}`.

In [3]:
cv_df = pd.read_csv(BASE_DIR / "reports/tables/stage5_cv_results.csv")
print("=== 5-FOLD CROSS-VALIDATION TRÊN TẬP TRAIN (N = 186,288) ===")
cv_cols = [
    "model_code",
    "cv_r2_mean",
    "cv_r2_std",
    "cv_rmse_mean",
    "cv_rmse_std",
    "cv_mae_mean",
    "cv_mape_mean",
    "mean_fold_time_sec",
]
print(cv_df[cv_cols].to_string(index=False))

=== 5-FOLD CROSS-VALIDATION TRÊN TẬP TRAIN (N = 186,288) ===
                    model_code  cv_r2_mean  cv_r2_std  cv_rmse_mean  cv_rmse_std  cv_mae_mean  cv_mape_mean  mean_fold_time_sec
      M0a_Tier_Median_Baseline      0.7628     0.0040        4.4866       0.0537       2.9648         19.29               0.046
M0b_Tier_Route_Lookup_Baseline      0.9148     0.0037        2.6881       0.0667       1.3036          8.32               0.194
                  M1_Basic_OLS      0.9254     0.0024        2.5160       0.0418       1.7456         12.87               0.550
            M2_Interaction_OLS      0.9539     0.0010        1.9785       0.0196       1.2859          9.36               0.810
      M2_Log_Duan_Smearing_OLS      0.9511     0.0009        2.0378       0.0167       1.3033          9.29               0.824
           M3_Ridge_Regression      0.9539     0.0010        1.9785       0.0196       1.2860          9.36               0.574
           M4_Lasso_Regression      0.9539 

In [4]:
test_df = pd.read_csv(BASE_DIR / "reports/tables/stage5_test_metrics.csv")
print("=== ĐÁNH GIÁ ĐỘC LẬP TRÊN TẬP TEST (N = 46,573 — SURGE THỰC) ===")
test_cols = [
    "model_code",
    "test_r2",
    "test_rmse",
    "test_mae",
    "test_mape_pct",
    "beats_M0b_baseline",
    "meets_both_smart",
    "train_time_sec",
]
print(test_df[test_cols].to_string(index=False))
ex_df = pd.read_csv(BASE_DIR / "reports/tables/stage5_ex_ante_experiments.csv")
print("\n=== THỰC NGHIỆM EX-ANTE (23 BIẾN KHÔNG DÙNG SURGE) ===")
print(ex_df.to_string(index=False))

=== ĐÁNH GIÁ ĐỘC LẬP TRÊN TẬP TEST (N = 46,573 — SURGE THỰC) ===
                    model_code  test_r2  test_rmse  test_mae  test_mape_pct  beats_M0b_baseline  meets_both_smart  train_time_sec
      M0a_Tier_Median_Baseline   0.7605     4.5148    2.9772          19.27               False             False           0.034
M0b_Tier_Route_Lookup_Baseline   0.9147     2.6944    1.3075           8.32               False              True           0.088
                  M1_Basic_OLS   0.9255     2.5182    1.7488          12.81                True             False           0.623
            M2_Interaction_OLS   0.9539     1.9802    1.2888           9.33                True              True           0.944
      M2_Log_Duan_Smearing_OLS   0.9515     2.0318    1.3037           9.25                True              True           0.957
           M3_Ridge_Regression   0.9539     1.9802    1.2889           9.33                True              True           0.621
           M4_Lasso_Regre

### Bước 2.2. Kiểm định Độ bền (Temporal Split, Route Group Split & Tập con Nguyên bản R3 `88.85%`)
- **Mục đích:** Chứng minh mô hình không bị suy sụp khi dự báo cho tương lai (`temporal_split`), tuyến đường mới chưa từng thấy (`route_group_split`), và hoàn toàn miễn nhiễm với nghi vấn rò rỉ từ bước khôi phục quy mô $10^m$ (`R3a` & `R3b`).


In [5]:
rob_df = pd.read_csv(BASE_DIR / "reports/tables/stage5_robustness.csv")
print(rob_df.to_string(index=False))

                                  scenario                     model_code                                    model_name  n_train  n_test     r2   rmse    mae  mape_pct
                            temporal_split M0b_Tier_Route_Lookup_Baseline     M0b: Bảng tra Gói xe x Tuyến (Baseline 2)   143023   35756 0.9158 2.6874 1.3030      8.27
                            temporal_split                   M1_Basic_OLS             M1: Hồi quy Tuyến tính OLS Cơ bản   143023   35756 0.9271 2.5008 1.7361     12.81
                            temporal_split             M2_Interaction_OLS       M2: Hồi quy OLS Tương tác Kinh tế lượng   143023   35756 0.9560 1.9421 1.2688      9.22
                            temporal_split            M3_Ridge_Regression         M3: Hồi quy Ridge (L2 Regularization)   143023   35756 0.9560 1.9421 1.2688      9.22
                            temporal_split               M6_Random_Forest     M6: Rừng Ngẫu nhiên (Random Forest Tuned)   143023   35756 0.9627 1.7877 1.1330   

### Bước 2.3. Sáu Thí Nghiệm Mutate & Break (MB1 -> MB6) & Quyết Định Chọn Mô Hình
- **Mô hình Dự báo Có điều kiện Chính (*Predictive Champion*):** `M7_HistGradientBoosting` ($R^2 = 0.9626$, $\text{RMSE} = \$1.7830$, $\text{MAPE} = 8.19\%$; Lyft $R^2 = 0.9794$, `1` lỗi $> \$15$ [`0` $> \$16$] vs Uber $R^2 = 0.9474$, `57` lỗi $> \$15$).
- **Mô hình Diễn giải Kinh tế lượng Chính (*Interpretability Champion*):** `M2_Interaction_OLS` ($R^2 = 0.9539$, $\text{RMSE} = \$1.9802$, $\text{MAPE} = 9.33\%$).
- **Đối chứng Ex-Ante (23 biến công khai, không dùng surge):** M7 đạt $R^2 = 0.9286$ ($\text{RMSE} = \$2.4655$; Lyft $>1.0\text{x}$: $R^2 = 0.4677$); bộ phân loại Surge Lyft đạt $\text{ROC-AUC} = 0.7383$, $\text{PR-AUC} = 0.1474$.

In [6]:
mb_df = pd.read_csv(BASE_DIR / "reports/tables/stage5_mutate_break.csv")
mb_cols = [
    "experiment_id",
    "baseline_r2",
    "mutated_r2",
    "baseline_rmse",
    "mutated_rmse",
    "baseline_mape_pct",
    "mutated_mape_pct",
    "scientific_lesson",
]
print(mb_df[mb_cols].to_string(index=False))

                             experiment_id  baseline_r2  mutated_r2  baseline_rmse  mutated_rmse  baseline_mape_pct  mutated_mape_pct                                                                                                                    scientific_lesson
        MB1_Drop_Tier_Distance_Interaction       0.9539      0.9255         1.9802        2.5182               9.33             12.81                               MAPE tăng vọt từ 9.33% lên 12.81% (+3.48%): chứng minh mỗi phân khúc xe có đơn giá mỗi dặm riêng biệt.
              MB2_Extreme_Ridge_L2_Penalty       0.9539      0.6438         1.9802        5.5061               9.33             36.52                         Hệ số hồi quy bị ép co mạnh về 0 gây Underfitting nặng (R2 tụt từ 0.9539 xuống 0.6438, RMSE tăng lên $5.51).
        MB3_Unconstrained_Tree_Overfitting       0.9580      0.9340         1.8898        2.3697               8.46              9.99     Cây học thuộc lòng nhiễu tập Train (Train R2=0.9911, RMSE=$0.